load and tokenize

In [1]:
from tqdm import tqdm
from transformers import AutoTokenizer
import os


tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [2]:
tokenizer.get_vocab

<bound method PreTrainedTokenizerFast.get_vocab of GPT2TokenizerFast(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)>

In [4]:
from tqdm import tqdm
from transformers import AutoTokenizer
import os


tokenizer = AutoTokenizer.from_pretrained("gpt2")

dir = 'arxiv_papers/txts'
paths = [os.path.join(dir, f) for f in os.listdir(dir)]
text = []
for p in paths:
    with open(p, 'r') as f:
        text.append(f.read())

tokenized_seq = []
for t in tqdm(text):
    tokenized_seq += tokenizer.encode(t)
len(tokenized_seq)


100%|██████████| 775/775 [00:11<00:00, 68.76it/s]


12962597

In [12]:
import random

ls = [1,2,3,4,5,6]
random.shuffle(ls)
ls

[1, 3, 5, 4, 6, 2]

In [ ]:
from pypdf import PdfReader
import os

directory = "transformer/papers"

text = ""

for filename in os.listdir(directory):

    path = os.path.join(directory, filename)

    if not filename.endswith(".pdf"):
        continue

    reader = PdfReader(path)

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text is None:
            continue

        page_text = page_text.encode(
            "utf-8",
            "ignore",
        ).decode("utf-8")

        text += page_text + "\n"

with open(
    "data.txt",
    "w",
    encoding="utf-8",
    errors="ignore",
) as f:

    f.write(text)

could not convert string to float: b'0.000-7375328' : FloatObject (b'0.000-7375328') invalid; use 0.0 instead


In [7]:
with open("data.txt", 'r') as f:
    data = f.readlines()

In [29]:
len(list(clean.split("transformer")))

222

In [1]:
import re

def clean_text(text):
    # Join lines that are broken mid-sentence (no punctuation at end)
    text = re.sub(r'(?<![.!?])\n(?!\n)', ' ', text)
    # Collapse multiple blank lines into one paragraph break
    text = re.sub(r'\n{2,}', '\n\n', text)
    # Remove weird unicode artifacts common in PDFs
    text = text.encode('utf-8', 'ignore').decode('utf-8')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

with open('data.txt', 'r', encoding='utf-8') as f:
    raw = f.read()

clean = clean_text(raw)

In [5]:
from transformers import AutoTokenizer

# Pick based on your model architecture:
tokenizer = AutoTokenizer.from_pretrained("gpt2")          # BPE, 50k vocab
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # WordPiece
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")  # SentencePiece

# token_ids = tokenizer.encode(clean)  # returns List[int]
# print(f"Total tokens: {len(token_ids):,}")  # expect ~3M+

In [2]:
corpus_splits = list(clean.split("transformer"))

In [6]:
tokenized_seq = []
for corp in corpus_splits:
    token_ids = tokenizer.encode(corp)
    tokenized_seq += token_ids


Token indices sequence length is longer than the specified maximum sequence length for this model (30583 > 1024). Running this sequence through the model will result in indexing errors


In [7]:
len(tokenized_seq)

905332

In [11]:
filtered_data = [f for f in data if len(f) > 1000]
len(filtered_data), len(data)

(0, 61720)

In [1]:
import torch

torch.rand((1, 512, 768))

tensor([[[0.4785, 0.7013, 0.2494,  ..., 0.1399, 0.1184, 0.4864],
         [0.9231, 0.1057, 0.6947,  ..., 0.1919, 0.4595, 0.4763],
         [0.3909, 0.3719, 0.6314,  ..., 0.2769, 0.6409, 0.0636],
         ...,
         [0.4004, 0.5454, 0.3395,  ..., 0.5465, 0.1607, 0.2053],
         [0.3743, 0.8309, 0.9140,  ..., 0.7931, 0.6338, 0.0254],
         [0.2739, 0.4838, 0.2694,  ..., 0.9548, 0.1725, 0.9198]]])

In [3]:
from bs4 import BeautifulSoup
import re

In [4]:

def remove_html(text):
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text(separator=" ")
    text = re.sub(r'\[\[.*?\]\]', '', text)       # wiki markup
    text = re.sub(r'\{\{.*?\}\}', '', text)       # templates
    text = re.sub(r'<[^>]+>', '', text)           # leftover tags
    return text


In [9]:
import unicodedata
import re

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)          # unicode normalize
    text = text.encode("utf-8", "ignore").decode("utf-8")  # fix encoding
    text = re.sub(r'[ \t]+', ' ', text)                 # collapse spaces
    text = re.sub(r'\n{3,}', '\n\n', text)              # max 2 newlines
    text = text.replace('\u201c', '"').replace('\u201d', '"')  # curly quotes
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    return text.strip()

In [14]:
with open('data.txt', 'r') as file:
    text = file.read()

print(text)
remove_html(text)
# normalize_text(text[700: 1000])

OCR-free Document Understanding Transformer
Geewook Kim1∗, Teakgyu Hong4†, Moonbin Yim2†, Jeongyeon Nam1,
Jinyoung Park5†, Jinyeong Yim6†, Wonseok Hwang7†, Sangdoo Yun3,
Dongyoon Han3, and Seunghyun Park1
1NAVER CLOVA 2NAVER Search 3NAVER AI Lab
4Upstage 5Tmax 6Google 7LBox
Abstract. Understanding document images (e.g., invoices) is a core but
challenging task since it requires complex functions such as reading text
and a holistic understanding of the document. Current Visual Document
Understanding (VDU) methods outsource the task of reading text to off-
the-shelf Optical Character Recognition (OCR) engines and focus on the
understanding task with the OCR outputs. Although such OCR-based
approaches have shown promising performance, they suffer from 1) high
computational costs for using OCR; 2) inflexibility of OCR models on
languages or types of documents; 3) OCR error propagation to the sub-
sequent process. To address these issues, in this paper, we introduce a
novel OCR-free VDU mod

'OCR-free Document Understanding Transformer\nGeewook Kim1∗, Teakgyu Hong4†, Moonbin Yim2†, Jeongyeon Nam1,\nJinyoung Park5†, Jinyeong Yim6†, Wonseok Hwang7†, Sangdoo Yun3,\nDongyoon Han3, and Seunghyun Park1\n1NAVER CLOVA 2NAVER Search 3NAVER AI Lab\n4Upstage 5Tmax 6Google 7LBox\nAbstract. Understanding document images (e.g., invoices) is a core but\nchallenging task since it requires complex functions such as reading text\nand a holistic understanding of the document. Current Visual Document\nUnderstanding (VDU) methods outsource the task of reading text to off-\nthe-shelf Optical Character Recognition (OCR) engines and focus on the\nunderstanding task with the OCR outputs. Although such OCR-based\napproaches have shown promising performance, they suffer from 1) high\ncomputational costs for using OCR; 2) inflexibility of OCR models on\nlanguages or types of documents; 3) OCR error propagation to the sub-\nsequent process. To address these issues, in this paper, we introduce a\nnovel

In [15]:
normalize_text(text[:100])


'OCR-free Document Understanding Transformer\nGeewook Kim1∗, Teakgyu Hong4†, Moonbin Yim2†, Jeongyeon'